# 🪐 Murray Buttes, Mars — Data Preparation & Exploration
### A reproducible, annotated NASA Space Apps notebook

**Study area:** Murray Buttes, Gale Crater, Mars  

## 1 · Setup and input data

This notebook builds the processed 1 m terrain table used by the routing notebook. Keep both notebooks in `Routing_model/notebooks/`, and place the input files in `Routing_model/data/raw/` (see the folder layout below).


```text
Routing_model/
├── README.md
├── notebooks/
│   ├── Murray_Buttes_Data_Preparation.ipynb
│   └── Murray_Buttes_Route_Planne.ipynb
├── data/
│   ├── raw/                         # input TIFFs, geology archive and artifact mask
│   ├── processed/                   # generated GeoTIFFs and geology layers
│   └── Murray_Buttes_Terrain_Data.csv  # final CSV, used by route planner
└── results/                        # routing figures and analysis results
```



In [17]:
from pathlib import Path
import re
import shutil
import zipfile

import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import (
    reproject, Resampling,
    transform as reproject_point,
    transform_bounds
)
from numpy.lib.stride_tricks import sliding_window_view
from scipy.ndimage import uniform_filter
import geopandas as gpd
from shapely.geometry import box

#SET WORKING DIRECTORY HERE!!!
#%cd "C:\Users\Admin\Downloads\Repository\notebooks"

# Notebook is inside notebooks/
# Go back one level to the project root
ROOT = Path("..").resolve()

RAW = ROOT / "data" / "raw"
DATA = ROOT / "data"
PROCESSED = DATA / "processed"
QGIS = PROCESSED / "rasters"
GEO = RAW

RAD = RAW / "radiation"    # Optional
ROVER = RAW / "rover"      # Optional
PACKAGE = PROCESSED / "supporting_data"


print("Current working directory:", Path.cwd())
print("Project root:", ROOT)
print("Raw data folder:", RAW)
print("Raw folder exists:", RAW.is_dir())

if not RAW.is_dir():
    raise FileNotFoundError(
        f"Cannot find raw data folder: {RAW}\n"
        "This relative path assumes Jupyter's working "
        "directory is notebooks/."
    )

# Create output folders
for folder in (DATA, PROCESSED, QGIS):
    folder.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "dem": RAW / "MSLGaleDEM_MAP2_EQUI.tif",
    "slope": RAW / "Murray Buttes Slope.tif",
    "image": RAW / "Murray Buttes Imagery.tif",
    "themis_raw": RAW / "Gale_crater_THEMIS_qualTI.tif",
    "geology_zip": RAW / "Murray Buttes GIS.zip",
    "artifact_mask": RAW / "Murray_Artifact_Suspected_Full.tif",
}


print("\nCHECKING INPUT FILES:")

missing_inputs = []

for name, path in SOURCES.items():
    if path.is_file():
        print(f"FOUND: {name} -> {path.name}")
    else:
        print(f"MISSING: {name} -> {path.name}")
        missing_inputs.append(str(path))

if missing_inputs:
    raise FileNotFoundError(
        "Missing required input files:\n"
        + "\n".join(missing_inputs)
    )

print("\nAll 6 input files found!")
print("Project root:", ROOT)
print("Master CSV output:",
      DATA / "Murray_Buttes_Terrain_Data.csv")
print("\nSetup completed successfully.")


Current working directory: C:\Users\Admin\Downloads\Repository\notebooks
Project root: C:\Users\Admin\Downloads\Repository
Raw data folder: C:\Users\Admin\Downloads\Repository\data\raw
Raw folder exists: True

CHECKING INPUT FILES:
FOUND: dem -> MSLGaleDEM_MAP2_EQUI.tif
FOUND: slope -> Murray Buttes Slope.tif
FOUND: image -> Murray Buttes Imagery.tif
FOUND: themis_raw -> Gale_crater_THEMIS_qualTI.tif
FOUND: geology_zip -> Murray Buttes GIS.zip
FOUND: artifact_mask -> Murray_Artifact_Suspected_Full.tif

All 6 input files found!
Project root: C:\Users\Admin\Downloads\Repository
Master CSV output: C:\Users\Admin\Downloads\Repository\data\Murray_Buttes_Terrain_Data.csv

Setup completed successfully.


## 2 · Inspect the DEM, slope and HiRISE imagery

Check that the DEM and slope raster have the same map geometry. HiRISE is 4 times finer (0.25 m rather than 1 m), but covers the same study extent. These assertions deliberately stop execution if the inputs are mismatched.

In [6]:
def describe_raster(path):
    with rasterio.open(path) as ds:
        print(f"{path.name}: {ds.shape}, {ds.count} band(s), {ds.res} m/pixel")
        print("  Bounds:", tuple(ds.bounds))
        print("  NoData:", ds.nodata)
        return {"shape": ds.shape, "res": ds.res, "bounds": ds.bounds,
                "transform": ds.transform, "crs": ds.crs}

with rasterio.open(SOURCES["dem"]) as ds:
    # Use the elevation band; additional bands are not elevations.
    dem = ds.read(1, masked=True).astype(np.float32).filled(np.nan)
    dem_profile, transform, crs, bounds = ds.profile.copy(), ds.transform, ds.crs, ds.bounds
with rasterio.open(SOURCES["slope"]) as ds:
    slope = ds.read(1, masked=True).astype(np.float32).filled(np.nan)
    # Slope and DEM must have matching grid geometry.
    assert ds.shape == dem.shape and ds.transform.almost_equals(transform)
    assert ds.crs == crs, "Slope CRS differs: Warning !! "

#using the function to print all of the neccessary data that we need
for path in (SOURCES["dem"], SOURCES["slope"], SOURCES["image"]):
    describe_raster(path)

with rasterio.open(SOURCES["image"]) as ds:
    # Three image bands; 0.25 m pixels give 4 x 4 pixels per 1 m DEM cell.
    assert ds.count >= 3 and ds.shape == (dem.shape[0] * 4, dem.shape[1] * 4)
    #We enforce a tolerance of 10^-5 between bounds
    assert np.allclose(ds.bounds, bounds, atol=1e-5), "Image extent differs"
    #calculating centers of the areas
    xc, yc = (bounds.left + bounds.right) / 2, (bounds.bottom + bounds.top) / 2
    #transforming DEM center data into HiRise CRS system to verfy similarity
    mapped_x, mapped_y = reproject_point(crs, ds.crs, [xc], [yc])
    #stricter conditoins
    assert np.allclose([mapped_x[0], mapped_y[0]], [xc, yc], atol=0.01)

print("\nDEM shape:", dem.shape, "| valid elevations:", np.isfinite(dem).sum())
print("Elevation range (m):", np.nanmin(dem), "to", np.nanmax(dem))
print("Slope range (degrees):", np.nanmin(slope), "to", np.nanmax(slope))
print("DEM and imagery share the same geographic extent in Mars metres.")

MSLGaleDEM_MAP2_EQUI.tif: (499, 498), 1 band(s), (1.0, 1.0) m/pixel
  Bounds: (8141308.0, -278556.0, 8141806.0, -278057.0)
  NoData: 0.0
Murray Buttes Slope.tif: (499, 498), 1 band(s), (1.0, 1.0) m/pixel
  Bounds: (8141308.0, -278556.0, 8141806.0, -278057.0)
  NoData: -9999.0
Murray Buttes Imagery.tif: (1996, 1992), 4 band(s), (0.25, 0.25) m/pixel
  Bounds: (8141308.0, -278556.0, 8141806.0, -278057.0)
  NoData: None

DEM shape: (499, 498) | valid elevations: 247506
Elevation range (m): -4402.52 to -4359.2437
Slope range (degrees): 0.02098234 to 59.975945
DEM and imagery share the same geographic extent in Mars metres.


## 3 · Derive terrain roughness from elevation

**Concept:** Slope measures the local *tilt* of the surface. Roughness measures the leftover bumps after subtracting a smooth best-fitting plane. On each valid 3 × 3 DEM window, we fit:

$$
\hat z(x,y) = a x + b y + c
\qquad\text{and}\qquad
R = \sqrt{\frac{1}{9}\sum_{i=1}^{9}(z_i-\hat z_i)^2}.
$$

The result is an RMS vertical deviation in metres, not a slope angle. The method cannot resolve features smaller than 1 m and necessarily leaves some boundary cells undefined (as discovered via EDA). sliding_window_view builds the overlapping neighborhoods; copying the DEM's GeoTIFF profile retains the map's CRS and pixel locations.

In [7]:
roughness_path = QGIS / "Murray_Buttes_Roughness.tif"

# 3x3 plane fit: arrays are 497 × 496 inside, then padded to 499 × 498.
w = sliding_window_view(dem.astype(np.float64), (3, 3))

# Use only windows with nine valid elevation samples.
valid = np.isfinite(w).all(axis=(-2, -1))
# Zero temporary invalid windows; put NaN back after the fit.
w = np.where(valid[..., None, None], w, 0.0)
# Local pixel coordinates, broadcast over all valid windows.
xlocal = np.arange(-1, 2)[None, :]
ylocal = np.arange(-1, 2)[:, None]

# Fit the local plane and calculate RMS residual roughness. 
a = (w * xlocal).sum(axis=(-2, -1)) / 6
b = (w * ylocal).sum(axis=(-2, -1)) / 6
c = w.mean(axis=(-2, -1))
plane = a[..., None, None] * xlocal + b[..., None, None] * ylocal + c[..., None, None]


inner = np.sqrt(np.mean((w - plane) ** 2, axis=(-2, -1)))

# Restore missing cells.
inner[~valid] = np.nan
roughness = np.full(dem.shape, np.nan, dtype=np.float32)
# The 3 x 3 method leaves a one-pixel border undefined.
roughness[1:-1, 1:-1] = inner.astype(np.float32)

profile = dem_profile.copy()
profile.update(driver="GTiff", count=1, dtype="float32", nodata=-9999.0, compress="lzw")
with rasterio.open(roughness_path, "w", **profile) as dst:
    dst.write(np.where(np.isfinite(roughness), roughness, -9999).astype(np.float32), 1)
print("Roughness saved:", roughness_path.name)
print("Valid:", np.isfinite(roughness).sum(), "| Median (m):", np.nanmedian(roughness))

Roughness saved: Murray_Buttes_Roughness.tif
Valid: 245520 | Median (m): 0.007892435


## 4 · Calculate HiRISE image texture

**Concept:** A color image contains intensity values, not height measurements. We convert RGB to *brightness*, then compute the standard deviation of brightness inside a 5 × 5 image-pixel window:

$$
I = 0.299R + 0.587G + 0.114B,
\qquad
T = \sqrt{E[I^2]-(E[I])^2}.
$$

At 0.25 m/pixel, the 5-pixel window spans approximately 1.25 m across. `uniform_filter` efficiently calculates the local means; `mode='reflect'` handles image edges. High texture can indicate rock, shadow, layer boundaries or lighting; it is not a hazard classification.

In [8]:
texture_path = QGIS / "Murray_Buttes_Texture.tif"
with rasterio.open(SOURCES["image"]) as ds:
    rgb = ds.read([1, 2, 3], masked=True).astype(np.float32).filled(np.nan)
    image_profile = ds.profile.copy()

# All pixels were valid in the project's cropped imagery; fail clearly otherwise.
if not np.isfinite(rgb).all():
    raise ValueError("HiRISE image contains missing RGB values; mask-aware texture needed")

# Standard RGB luminance weighting.
brightness = .299 * rgb[0] + .587 * rgb[1] + .114 * rgb[2]
# Local means over a 5 x 5 HiRISE pixel window.
mu = uniform_filter(brightness, size=5, mode="reflect")
mu2 = uniform_filter(brightness ** 2, size=5, mode="reflect")

# Texture is the local standard deviation of brightness.
texture = np.sqrt(np.maximum(mu2 - mu ** 2, 0)).astype(np.float32)

# Export a single-band texture GeoTIFF on the image grid.
image_profile.update(driver="GTiff", count=1, dtype="float32", nodata=-9999.0,
                     compress="lzw", photometric="MINISBLACK")
with rasterio.open(texture_path, "w", **image_profile) as dst:
    dst.write(texture, 1)
print("Texture saved:", texture_path.name, "| shape:", texture.shape)
print("Range:", float(texture.min()), "to", float(texture.max()))

Texture saved: Murray_Buttes_Texture.tif | shape: (1996, 1992)
Range: 0.0 to 121.88976287841797


## 5 · Geological vectors: extract, clip and package

The geological archive contains mapped vector layers. We clip those layers to the DEM study extent and save a GeoPackage for later joining. **Mars CRS note:** some source shapefiles use unusual or incomplete CRS metadata. The original workflow assumed the source coordinate numbers and DEM coordinates already correspond; below we check extents before clipping. Do not blindly reinterpret a source with genuinely different coordinates.

In [9]:
archive = SOURCES["geology_zip"]
extracted = PROCESSED / "Murray_Buttes_Extracted"
if not extracted.exists():
    # Avoid extracting unsafe ZIP paths outside the intended folder.
    with zipfile.ZipFile(archive) as zf:
        dest = extracted.resolve()
        for member in zf.infolist():
            if not (dest / member.filename).resolve().is_relative_to(dest):
                raise ValueError(f"Unsafe path in geology archive: {member.filename}")
        zf.extractall(extracted)

shapefiles = sorted(extracted.rglob("*.shp"))
if not shapefiles:
    raise FileNotFoundError(f"No shapefiles found in {archive}")

study_poly = box(*bounds)
geology_path = PROCESSED / "Murray_Buttes_Geology.gpkg"
if geology_path.exists():
    geology_path.unlink()  # Rebuild from the supplied archive.

written_layers = []
for shp in shapefiles:
    source = gpd.read_file(shp)
    if source.crs is None:
        raise ValueError(f"CRS missing in {shp.name}; check the source metadata")

    # Original data were aligned in projected Mars metres. Check for spatial overlap.
    # This is not a generic CRS conversion and must be revisited for other GIS inputs.
    matching = source[source.geometry.intersects(study_poly)].copy()
    if matching.empty:
        continue

    matching.geometry = matching.geometry.intersection(study_poly)
    layer_name = shp.stem.replace(" ", "_")
    matching.to_file(geology_path, layer=layer_name, driver="GPKG")
    written_layers.append(layer_name)
    print(f"{layer_name}: {len(matching)} features")

if "JVB_MurrayButtes" not in written_layers:
    raise ValueError("JVB_MurrayButtes layer was not produced; check the geology archive/CRS")
print("Geology saved:", geology_path)

Butte_spot: 10 features
Butte_Spotheight: 10 features
JVB_MurrayButtes: 10 features
MB_AOI: 1 features
Mosaic_View: 5 features
PC_CrossLam: 53 features
PC_Superposition: 11 features
PDSRoverLoc: 16 features
Geology saved: C:\Users\Admin\Downloads\Repository\data\processed\Murray_Buttes_Geology.gpkg


## 6· THEMIS: check the CRS and align the coarse raster

In [10]:
themis_path = SOURCES["themis_raw"]
with rasterio.open(themis_path) as ds:
    if ds.crs is None:
        raise ValueError("THEMIS CRS missing")

    # Verify spatial overlap after transforming the DEM bounds to the THEMIS CRS.
    transformed = transform_bounds(crs, ds.crs, *bounds, densify_pts=21)
    overlap = (transformed[0] < ds.bounds.right and transformed[2] > ds.bounds.left
               and transformed[1] < ds.bounds.top and transformed[3] > ds.bounds.bottom)
    if not overlap:
        raise ValueError("THEMIS extent does not overlap study area after CRS transformation")

    # Nearest-neighbor resampling preserves the source's qualitative pixel codes.
    # 1 m output does not imply the original measurement has 1 m resolution.
    themis_1m = np.zeros(dem.shape, dtype=np.uint8)
    reproject(source=rasterio.band(ds, 1), destination=themis_1m,
              src_transform=ds.transform, src_crs=ds.crs, src_nodata=ds.nodata,
              dst_transform=transform, dst_crs=crs, dst_nodata=0,
              resampling=Resampling.nearest)

aligned_themis = QGIS / "Murray_Buttes_THEMIS_1m.tif"
profile = dem_profile.copy()
profile.update(driver="GTiff", count=1, dtype="uint8", nodata=0, compress="lzw")
with rasterio.open(aligned_themis, "w", **profile) as ds:
    ds.write(themis_1m, 1)
print("THEMIS coverage:", np.count_nonzero(themis_1m), "/", themis_1m.size)
print("Unique nonzero source values:", len(np.unique(themis_1m[themis_1m != 0])))

THEMIS coverage: 248502 / 248502
Unique nonzero source values: 24


## 7 · Build the master terrain table (one row per 1 m cell)

**Concept:** A raster is a 2D array. A machine-readable **master CSV** is a *long table*: each raster cell becomes one row and each aligned terrain factor becomes a column.

The DEM's **affine transform** provides spatial coordinates for the *centres* of pixels:

\[
X = c + (\mathrm{col}+0.5)a + (\mathrm{row}+0.5)b,\qquad
Y = f + (\mathrm{col}+0.5)d + (\mathrm{row}+0.5)e.
\]

This vectorized NumPy calculation avoids the expensive `rasterio.transform.xy()` call that may have contributed to a previous kernel crash. Convert integer arrays to floating point before using `NaN` (the earlier THEMIS error was caused by trying to fill a `uint8` array with `NaN`).

In [11]:
rows, cols = np.indices(dem.shape)

# Convert raster row/column indices to Mars-projected cell-centre coordinates.  
x = transform.c + (cols + .5) * transform.a + (rows + .5) * transform.b
y = transform.f + (cols + .5) * transform.d + (rows + .5) * transform.e

master = pd.DataFrame({
    "row": rows.ravel(), "col": cols.ravel(),
    "x_mars": x.ravel(), "y_mars": y.ravel(),
    "elevation_m": dem.ravel(), "slope_deg": slope.ravel(),
    "roughness_m": roughness.ravel(),
    "themis_qualitative": np.where(themis_1m == 0, np.nan, themis_1m).ravel(),
})
master["valid_terrain"] = master[["elevation_m", "slope_deg", "roughness_m"]].notna().all(axis=1)
print("Rows:", len(master), "| columns so far:", master.shape[1])
print(master.isna().sum())

Rows: 248502 | columns so far: 9
row                      0
col                      0
x_mars                   0
y_mars                   0
elevation_m            996
slope_deg              996
roughness_m           2982
themis_qualitative       0
valid_terrain            0
dtype: int64


## 8 · Audit missing values (do not invent DEM measurements)

This checks missing DEM and derivative values. Note that the routing notebook **separately** reconstructs some flagged artifact cells *in memory* and warns when a route crosses them. The master CSV created here retains the raw DEM elevations.

In [12]:
for col in ("elevation_m", "slope_deg", "roughness_m"):
    missing = master[col].isna().to_numpy().reshape(dem.shape)
    print(f"{col}: {missing.sum()} missing; full rows {np.where(missing.all(axis=1))[0].tolist()}; "
          f"full cols {np.where(missing.all(axis=0))[0].tolist()}")
print("Valid terrain cells:", int(master.valid_terrain.sum()))
print("Valid terrain percent:", round(master.valid_terrain.mean() * 100, 2))

elevation_m: 996 missing; full rows [0]; full cols [0]
slope_deg: 996 missing; full rows [0]; full cols [0]
roughness_m: 2982 missing; full rows [0, 1, 498]; full cols [0, 1, 497]
Valid terrain cells: 245520
Valid terrain percent: 98.8


## 9 · Aggregate HiRISE texture and RGB to the 1 m grid

Each 1 m DEM cell corresponds to 4 × 4 HiRISE pixels in these already aligned inputs. Compute the mean and maximum texture, and average each RGB band. These are descriptive image features, not directly verified hazard measurements.

In [13]:
h, w = dem.shape
# Use NaN-compatible arrays for any masked pixels.
texture_blocks = texture.reshape(h, 4, w, 4)
master["texture_mean"] = np.nanmean(texture_blocks, axis=(1, 3)).ravel()
master["texture_max"] = np.nanmax(texture_blocks, axis=(1, 3)).ravel()

rgb_blocks = rgb.reshape(3, h, 4, w, 4)
color_means = np.nanmean(rgb_blocks, axis=(2, 4))
for band_index, color in enumerate(("red", "green", "blue")):
    master[f"{color}_mean"] = color_means[band_index].ravel()
print("Mean/max texture correlation:", master[["texture_mean", "texture_max"]].corr().iloc[0, 1])

Mean/max texture correlation: 0.9511764070980098


## 10 · Mark cells inside the mapped Murray Buttes polygons

In [14]:
buttes = gpd.read_file(geology_path, layer="JVB_MurrayButtes")
assert "Feature_No" in buttes.columns

# The project source polygon coordinates were checked against the DEM grid.
# A CRS metadata override is valid only for this previously verified source.
if not np.isfinite(buttes.total_bounds).all():
    raise ValueError("Invalid butte polygon coordinates")
buttes = buttes.set_crs(crs, allow_override=True)

cells = gpd.GeoDataFrame(master[["row", "col"]].copy(),
                         geometry=gpd.points_from_xy(master.x_mars, master.y_mars), crs=crs)
joined = gpd.sjoin(cells, buttes[["Feature_No", "geometry"]],
                   how="left", predicate="within")
assert len(joined) == len(master), "Overlapping polygon matches: resolve before merging"
joined = joined.sort_index()
master["butte_id"] = joined["Feature_No"].to_numpy()
master["inside_butte"] = master.butte_id.notna()
print("Mapped buttes:", master.butte_id.nunique(), "| cells inside:", int(master.inside_butte.sum()))

Mapped buttes: 10 | cells inside: 25272


## 11 · Add the suspected DEM artifact mask

The final routing notebook expects `artifact_suspected` for each 1 m cell. That column was missing from the original preprocessing notebook. It must come from the **existing artifact mask** (`Murray_Artifact_Suspected_Full.tif`) rather than inventing a new classification from image brightness or elevation alone. We check the mask's shape, transform, and CRS before copying its flags into the table.

In [15]:
artifact_mask_path = SOURCES["artifact_mask"]
with rasterio.open(artifact_mask_path) as ds:
    if ds.shape != dem.shape or not ds.transform.almost_equals(transform) or ds.crs != crs:
        raise ValueError("Artifact mask is not aligned with the DEM grid")
    mask_data = ds.read(1, masked=True)
    artifact_suspected = np.asarray(mask_data.filled(0) > 0, dtype=bool)

master["artifact_suspected"] = artifact_suspected.ravel()
print("Suspected DEM artifact cells:", int(master["artifact_suspected"].sum()))

# Keep the original measured DEM values intact in this preprocessing stage.
# The downstream route planner may interpolate flagged cells in memory,
# but those reconstructed elevations should not overwrite the master file.

Suspected DEM artifact cells: 1553


## 12 · Validate and export the final master terrain CSV

Check the complete schema, grid indexing, numerical coverage, and artifact flags before saving. The required output is `data/Murray_Buttes_Terrain_Data.csv`, which the routing notebook should load. Extra processed GeoTIFFs and geology layers are saved under `data/processed/`.

Radiation/RAD and rover-position files are independent contextual datasets, so they are not added as invented per-cell values to the 1 m terrain table. Optional copies of those inputs are handled below when present.

In [16]:
# Check that all fields needed by the route planner are present.
expected_columns = [
    "row", "col", "x_mars", "y_mars", "elevation_m", "slope_deg",
    "roughness_m", "themis_qualitative", "valid_terrain",
    "texture_mean", "texture_max", "butte_id", "inside_butte",
    "red_mean", "green_mean", "blue_mean", "artifact_suspected"
]
missing_columns = sorted(set(expected_columns) - set(master.columns))
if missing_columns:
    raise ValueError(f"Missing master CSV columns: {missing_columns}")
master = master[expected_columns].copy()

expected_rows = dem.shape[0] * dem.shape[1]
assert len(master) == expected_rows, "Master has an unexpected number of cells"
assert not master.duplicated(["row", "col"]).any(), "Duplicate raster cells"
assert master["row"].between(0, dem.shape[0] - 1).all()
assert master["col"].between(0, dem.shape[1] - 1).all()
assert master["artifact_suspected"].dtype == bool
assert master["valid_terrain"].equals(
    master[["elevation_m", "slope_deg", "roughness_m"]].notna().all(axis=1)
)

# Use a predictable filename for the GitHub route-planning notebook.
master_path = DATA / "Murray_Buttes_Terrain_Data.csv"
master.to_csv(master_path, index=False)

# Read it back to catch write/format issues rather than assuming a successful export.
check = pd.read_csv(master_path)
assert check.shape == master.shape, "Saved CSV shape differs"
assert list(check.columns) == expected_columns, "Saved CSV fields differ"
assert int(check["artifact_suspected"].sum()) == int(master["artifact_suspected"].sum())

print("Master table:", master_path)
print("Shape:", master.shape)
print("Valid terrain cells:", int(master.valid_terrain.sum()))
print("Flagged artifact cells:", int(master.artifact_suspected.sum()))
print("Processed rasters:", QGIS)
print("Geology GeoPackage:", geology_path)

# Optional: collect supporting radiation/rover files if they exist.
# These are *not* required to build the master terrain CSV.
COPY_SUPPORTING_FILES = False
if COPY_SUPPORTING_FILES:
    folders = {name: PACKAGE / name for name in ("Radiation", "Rover")}
    for folder in folders.values():
        folder.mkdir(parents=True, exist_ok=True)
    for src_dir, dest_dir in ((RAD, folders["Radiation"]),
                              (ROVER, folders["Rover"])):
        if src_dir.is_dir():
            for src in src_dir.rglob("*"):
                if src.is_file():
                    dest = dest_dir / src.relative_to(src_dir)
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(src, dest)
    print("Optional supporting data copied to:", PACKAGE)

Master table: C:\Users\Admin\Downloads\Repository\data\Murray_Buttes_Terrain_Data.csv
Shape: (248502, 17)
Valid terrain cells: 245520
Flagged artifact cells: 1553
Processed rasters: C:\Users\Admin\Downloads\Repository\data\processed\rasters
Geology GeoPackage: C:\Users\Admin\Downloads\Repository\data\processed\Murray_Buttes_Geology.gpkg


C:\Users\Admin\AppData\Local\Temp\ipykernel_42444\2375305487.py:28: DtypeWarning: Columns (11: butte_id) have mixed types. Specify dtype option on import or set low_memory=False.
  check = pd.read_csv(master_path)
